# Blob pseudo-labels for synaptic puncta -- real data

Build **binary pseudo-labels** for synaptic puncta on real 3-channel
128x128 fluorescence patches loaded via `PatchDataset` (no manual
annotation). Output is a `(1, 128, 128)` float32 mask per patch
(1 = synapse, 0 = background) usable as a downstream segmentation
target.

> **Channel convention**: ch0 = pre-synaptic, ch1 = post-synaptic,
> ch2 = structural / neurite marker.

> **Pixel size assumed**: 107 nm/px (confocal, 60x objective).
> Puncta diameter is 2-5 px; dendrite diameter 4-18 px.

**Visualisation strategy** (matches the from-scratch training
templates):

* **Minimal** per-stage figures use the canonical
  `SANITY_TRAIN_INDICES = (3, 4, 50, 200, 400)` -- the same patches
  used for the overfit sanity check across every training notebook,
  so per-stage figures are directly comparable to recon panels.
* **After full run** the visual review and aggregate-statistic
  figures are computed over many patches (`N_VISUAL_REVIEW = 24`).
* Every figure is saved to `save_dir/figures/` for the thesis.


## What this notebook does

We chain five image-processing operations and a co-localization rule
to turn raw fluorescence into a binary pseudo-label:

1. **Structural mask** (where neurons live) -- Meijering ridge filter
   for dendrites + intensity threshold for somas, on the structural
   channel.
2. **Blob detection** -- scale-space Laplacian-of-Gaussian (LoG) on
   the pre and post channels, separately.
3. **Co-localization** -- pre and post blobs must overlap (with a
   2 px tolerance for sub-pixel registration).
4. **z-score filter** -- SynQuant-lite test against a local annular
   background.
5. **Shape priors** + **restrict to structural mask** -- keep blobs
   that look like puncta and lie on or near a neuron.


## Background

### Blob detection (Laplacian-of-Gaussian, LoG)

A *blob* is a bright spot of roughly known size on a darker
background. `skimage.feature.blob_log` runs a multi-scale LoG search
and returns one row per detection: `(row, col, sigma)`. For our
puncta (2-5 px diameter, radius 1-2.5 px) we use
`min_sigma=0.7, max_sigma=1.8` (Lindeberg 1998).

### Ridge filters (Meijering)

A *ridge* is a thin elongated bright structure. The Meijering filter
(Meijering et al. 2004) was designed for fluorescent neurite
tracing. For our dendrites (radius 2-9 px) we use
`sigmas=range(2, 10)`.

### Why a structural mask + co-localization + z-score?

Real synapses sit on dendrites/somas, not in random debris. Real
synapses have BOTH pre and post markers within ~200 nm. Faint LoG
candidates indistinguishable from noise must be filtered. Each step
removes a different class of false positive.


## Patch-mode caveats

Most blob/ridge papers operate on full-resolution images (1024+ px).
We operate on **128x128 patches** for downstream training, but the
structural detectors (Meijering ridge filter for dendrites + intensity
threshold for somas) are run on the **full reassembled image** and
then sliced per patch. This avoids the two pitfalls below; only the
small-scale LoG step is genuinely safe at patch level.

| Issue | Effect on 128x128 | Fix in this notebook |
|---|---|---|
| `blob_log` reflects pixels at borders inside its Gaussian-Laplace convolution; LoG response is attenuated within ~3*sigma_max of the edge | At sigma_max=1.8 px we lose detection in a ~5-px-wide rim, ~15% of patch area | `cfg.log_exclude_border = 5` -- documented loss |
| Meijering ridge filter has border artefacts up to ~3*sigma_max wide; with `sigmas=range(2, 10)` that is ~27 px per side -- ~42% of a 128 px patch is degraded | Dendrites are missed near every patch edge | **Run Meijering on the full reassembled image** via `compute_fullimage_structural_mask`, then slice per patch |
| `threshold_otsu` is a global, histogram-based statistic. Per-patch Otsu on the Meijering response is unreliable for patches without dendrites (unimodal histograms) | Threshold randomly fluctuates across patches | Pool full-image Meijering responses across `N_CALIBRATION_IMAGES` images and compute **one global threshold** (`compute_global_meijering_threshold`) |
| Somas (~25-50 px diameter) that straddle a patch boundary are fragmented and may fall under `soma_min_area` | Partial somas are dropped | Detect somas on the full image so each soma is one connected component |
| The annular z-score needs an annulus that fits inside the image | Inner=3, outer=8 -> 17 px window, fits | No change -- z-score stays per-patch |

Bottom line: dendrites and somas are detected on the full image;
only the small-scale LoG + co-localisation + z-score run per patch.


## Imports


In [ ]:
import os, sys, json
from datetime import datetime
from pathlib import Path


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


## Path setup


In [ ]:
NB_DIR = Path.cwd().resolve()
NB_NEW = NB_DIR.parent if NB_DIR.parent.name == 'notebooks' else NB_DIR.parents[1]
ROOT   = NB_NEW.parent
for p in (NB_NEW, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('NB_NEW:', NB_NEW)
print('ROOT  :', ROOT)


In [ ]:
from pseudolabels.blobs import (
    BlobPseudoCfg,
    compute_global_meijering_threshold,
    compute_fullimage_structural_mask,
    meijering_response,
    detect_blobs_log,
    score_blobs_zscore,
    generate_blob_pseudolabel,
)
from pseudolabels.viz import (
    show_3channel_grid,
    show_blob_overlay,
    show_scored_blobs,
    show_mask_overlay,
    show_pipeline_stages,
    plot_zscore_histogram,
)
from training.sanity_batch import SANITY_TRAIN_INDICES
from training.logging import setup_logger
from utils_data.reassemble import reassemble_image
from collections import defaultdict
from matplotlib.patches import Rectangle


## Configuration


In [ ]:
# Data + IO
PATCH_ROOT  = ROOT / '..' / 'data' / 'patches_128'
OUTPUT_ROOT = ROOT / '..' / 'data' / 'pseudolabels_blob'
EXCLUDE_PATTERNS = ['KONTROLA']  # case-insensitive substrings to skip

# Sample sizes
N_CALIBRATION_IMAGES  = 8     # full images pooled for the Meijering threshold
N_VISUAL_REVIEW       = 24    # post-run review grid (multiple of 4)
SEED                  = 42

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('PATCH_ROOT :', PATCH_ROOT, ' (exists:', PATCH_ROOT.exists(), ')')
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
# All knobs in one place. Defaults are calibrated for 107 nm/px confocal,
# puncta diameter 2-5 px, dendrite diameter 4-18 px.

cfg = BlobPseudoCfg(
    pre_channel=0, post_channel=1, structural_channel=2,

    # LoG on pre/post (puncta detection)
    log_min_sigma=0.7,
    log_max_sigma=1.8,
    log_num_sigma=5,
    log_threshold=0.005,    # very permissive; the z-score does the real filtering
    log_overlap=0.5,
    log_exclude_border=5,

    # co-localization tolerance (sub-pixel mis-registration)
    coloc_dilation=2,

    # Meijering for dendrites
    dendrite_sigmas=list(range(2, 10)),
    dendrite_threshold=None,  # filled in by the calibration cell below

    # soma (intensity threshold + size filter)
    soma_intensity_percentile=90.0,  # lowered from 99 to catch dimmer somas
    soma_min_area=2500,       # lowered from 5000 to catch smaller somas
                              # (real somas are 10-30 um diameter,
                              #  i.e. ~7000-60000 px^2; smaller values
                              #  may include bright dendrite junctions)
    # Optional morphological polish: closing consolidates fragmented
    # bright regions inside one soma; fill_holes plugs the dim nucleolus.
    # Leave OFF if the raw threshold + min_area already gives clean blobs
    # (check `04b_soma_percentile_sweep`). Enable only if you see one
    # obvious soma getting broken into several small CCs below min_area.
    soma_closing_radius=0,
    soma_fill_holes=False,

    # near-neuron zone width
    structural_dilation=4,    # ~ 0.43 um

    # shape priors
    min_size=3, max_size=40,
    min_fill=0.5, max_wh_ratio=4.0,

    # SynQuant-lite z-score
    use_zscore=True,
    zscore_inner_radius=3,
    zscore_outer_radius=8,
    zscore_threshold=5.0,     # 5-7 = friendly recall; SynQuant default = 10 (strict)
)
print(cfg)


## Output directory and logger


In [ ]:
RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = NB_DIR / 'outputs' / f'blob_pseudolabels_{RUN_TS}'
fig_dir  = save_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)
print('fig_dir  =', fig_dir)


In [ ]:
logger = setup_logger('nb', save_dir / 'run.log')
logger.info(f'experiment = blob_pseudolabels')
logger.info(f'save_dir   = {save_dir}')
logger.info(f'patch_root = {PATCH_ROOT}  (exists={PATCH_ROOT.exists()})')
logger.info(f'output_root= {OUTPUT_ROOT}')


In [ ]:
# Persist the configuration alongside the figures so any thesis figure
# can be traced back to the exact parameters that produced it.
(save_dir / 'config.json').write_text(
    json.dumps(
        {
            'cfg': cfg.__dict__,
            'patch_root': str(PATCH_ROOT),
            'output_root': str(OUTPUT_ROOT),
            'exclude_patterns': EXCLUDE_PATTERNS,
            'n_calibration_images': N_CALIBRATION_IMAGES,
            'n_visual_review': N_VISUAL_REVIEW,
            'sanity_train_indices': list(SANITY_TRAIN_INDICES),
            'seed': SEED,
        },
        indent=2, default=str,
    )
)
logger.info('config.json written')


## Plot saving helper

Every figure goes through `save_fig(fig, name)` which writes
`save_dir/figures/<name>.png` at 200 dpi. This keeps the thesis-ready
outputs co-located with the run log and config.


In [ ]:
def save_fig(fig, name: str, dpi: int = 200, close: bool = False):
    """Save `fig` to ``fig_dir/<name>.png``. Returns the resolved path.

    All thesis figures go through here so that the filename, dpi and
    bbox handling are consistent. Call ``plt.show()`` separately if you
    want the figure rendered in the notebook output (default keeps it
    open so the cell that follows can still call ``plt.show()``).
    """
    out = fig_dir / f'{name}.png'
    fig.savefig(out, dpi=dpi, bbox_inches='tight')
    logger.info(f'[fig] {out.relative_to(save_dir)}')
    if close:
        plt.close(fig)
    return out


## Seed


In [ ]:
np.random.seed(SEED)
logger.info(f'seed = {SEED}')


## Data

Real patches are loaded via `PatchDataset` (the same class used by
all training notebooks). Each item is `(C, H, W)` float32 in `[0, 1]`.
If `PATCH_ROOT` is not mounted we fall back to a small synthetic set
so the notebook still runs end-to-end -- the parameter calibration
below assumes real data, so synthetic results are diagnostic only.


In [ ]:
def make_synthetic_patch(rng, n_pre=20, n_post=20, n_dendrite=3, soma=True):
    """Fallback generator -- only used when the real patch dir is missing."""
    H = W = 128
    img = np.zeros((3, H, W), dtype=np.float32)
    img += rng.normal(0.05, 0.01, img.shape).clip(0, None)
    for _ in range(n_dendrite):
        y0 = rng.integers(20, 100); x0 = rng.integers(20, 100)
        dy, dx = rng.normal(0, 1), rng.normal(0, 1)
        n = np.hypot(dy, dx) + 1e-6; dy/=n; dx/=n
        for t in range(70):
            yy, xx = int(y0 + t*dy), int(x0 + t*dx)
            if 1 <= yy < H-1 and 1 <= xx < W-1:
                img[2, yy-1:yy+2, xx-1:xx+2] += 0.4
    if soma:
        cy, cx = 25, 25
        yy, xx = np.mgrid[:H, :W]
        img[2] += 0.9 * np.exp(-((yy-cy)**2 + (xx-cx)**2) / (2 * 9**2))
    for _ in range(n_pre):
        y, x = rng.integers(8, 120), rng.integers(8, 120)
        sy, ey = max(0, y-2), min(H, y+3); sx, ex = max(0, x-2), min(W, x+3)
        img[0, sy:ey, sx:ex] += 0.7
        oy = y + rng.integers(-1, 2); ox = x + rng.integers(-1, 2)
        sy, ey = max(0, oy-2), min(H, oy+3); sx, ex = max(0, ox-2), min(W, ox+3)
        img[1, sy:ey, sx:ex] += 0.7
    for _ in range(10):
        y, x = rng.integers(8, 120), rng.integers(8, 120)
        sy, ey = max(0, y-2), min(H, y+3); sx, ex = max(0, x-2), min(W, x+3)
        img[0, sy:ey, sx:ex] += 0.4
    return img.clip(0, 1)


In [ ]:
import csv

def load_patch_records(patch_root: Path, exclude_patterns):
    """Load only the index.csv records (no pixel data).

    Each record carries ``filename``, ``source_image``, ``image_index``,
    ``grid_row``, ``grid_col``, ``patch_size``, ``channels``.  Patch
    pixels are sliced on demand from cached reassembled images via
    ``get_patch`` (defined in the next cell), so opening the dataset is
    O(rows in CSV) instead of one np.load() per patch.
    """
    csv_path = Path(patch_root) / 'index.csv'
    with open(csv_path) as f:
        records = list(csv.DictReader(f))
    pats = [p.upper() for p in (exclude_patterns or [])]
    records = [
        r for r in records
        if str(r.get('damaged', '')).strip().lower() not in ('true', '1')
        and not any(p in r['source_image'].upper() for p in pats)
    ]
    return records


if PATCH_ROOT.exists():
    patch_records = load_patch_records(PATCH_ROOT, EXCLUDE_PATTERNS)
    logger.info(f'indexed {len(patch_records)} real patches from {PATCH_ROOT}')



In [ ]:
# Resolve the canonical sanity indices against the actual dataset size
# (the helper caps any out-of-range index to len(dataset) - 1, matching
# training.sanity_batch._safe_indices).
SANITY_INDICES = [min(int(i), len(patch_records) - 1) for i in SANITY_TRAIN_INDICES]
logger.info(f'SANITY_TRAIN_INDICES (resolved) = {SANITY_INDICES}')
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    logger.info(f'  sanity[{j}] idx={i:5d}  {name}')


## Full-image structural mask cache

Dendrite (Meijering) and soma detection are run on the **full
reassembled image** (typically 2304x2304) so that ridge-filter
border artefacts and split somas do not bias the structural mask.
Each full-image structural mask is computed once per image and
sliced back into 128x128 tiles aligned with the patch grid.

The cache lazily reassembles each image only when first requested,
so calibration / demos only touch the few images they need.


In [ ]:
from collections import defaultdict
from utils_data.reassemble import reassemble_image

# patch position -> source-image grouping
image_to_patch_positions: dict[int, list[int]] = defaultdict(list)
for pos, rec in enumerate(patch_records):
    image_to_patch_positions[int(rec['image_index'])].append(pos)
available_image_indices = sorted(image_to_patch_positions)
logger.info(
    f'{len(available_image_indices)} unique source images cover '
    f'{len(patch_records)} patches'
)

_image_cache: dict[int, np.ndarray] = {}
_struct_full_cache: dict[int, dict] = {}


def get_full_image(image_index: int) -> np.ndarray:
    """Reassemble a full ``(C, H, W)`` image once and cache it.

    Replaces the per-patch ``np.load`` loop -- one read per patch file,
    one reassembly per image (instead of two: load + reassemble).
    """
    if image_index in _image_cache:
        return _image_cache[image_index]
    full_image, _ = reassemble_image(PATCH_ROOT, image_index)
    _image_cache[image_index] = full_image
    return full_image


def get_patch(pos: int) -> np.ndarray:
    """Slice one ``(C, ps, ps)`` patch out of its cached full image."""
    rec = patch_records[pos]
    full = get_full_image(int(rec['image_index']))
    ps = int(rec['patch_size'])
    y0 = int(rec['grid_row']) * ps
    x0 = int(rec['grid_col']) * ps
    return full[:, y0:y0 + ps, x0:x0 + ps]


def get_full_struct(image_index: int) -> dict:
    """Return the full-image structural dict for one source image.

    Returns a dict with keys ``meijering_response``, ``dendrite_mask``,
    ``soma_mask``, ``structural_mask``, ``near_structural`` (each
    ``(H_full, W_full)``) plus ``full_image``.  Reassembles + computes
    on first access; cached afterwards.

    Requires ``cfg.dendrite_threshold`` to be set (run the calibration
    cell first).
    """
    if image_index in _struct_full_cache:
        return _struct_full_cache[image_index]
    full_image = get_full_image(image_index)
    struct = compute_fullimage_structural_mask(full_image, cfg)
    struct['full_image'] = full_image
    _struct_full_cache[image_index] = struct
    return struct

def get_patch_struct_slice(pos: int) -> dict:
    """Slice the cached full-image structural mask for one patch position."""
    rec = patch_records[pos]
    image_index = int(rec['image_index'])
    grid_row = int(rec['grid_row'])
    grid_col = int(rec['grid_col'])
    patch_size = int(rec['patch_size'])
    full = get_full_struct(image_index)
    y0 = grid_row * patch_size
    x0 = grid_col * patch_size
    sl = (slice(y0, y0 + patch_size), slice(x0, x0 + patch_size))
    return {
        'meijering_response': full['meijering_response'][sl],
        'dendrite_mask':      full['dendrite_mask'][sl],
        'soma_mask':          full['soma_mask'][sl],
        'structural_mask':    full['structural_mask'][sl],
        'near_structural':    full['near_structural'][sl],
    }


## Sanity checks

Minimal visualisations on the canonical sanity batch (`SANITY_TRAIN_INDICES`).
These figures answer two questions before any pipeline parameters are
calibrated:

1. Is the channel order correct (pre/post sparse dots, structural
   dendrite + soma morphology)?
2. Is the percentile normalisation correct (most signal in `[0, 0.3]`,
   sparse high-intensity tail)?


In [ ]:
# Per-patch 3-channel grids on every sanity-batch index.
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    p = get_patch(i)
    print(f'sanity[{j}]  idx={i}  {name}  range=[{p.min():.3f}, {p.max():.3f}]')
    fig, _ = show_3channel_grid(p)
    fig.suptitle(f'sanity[{j}]  idx={i}  {name}', y=1.02, fontsize=10)
    save_fig(fig, f'01_sanity_3channel_idx{i:05d}')
    plt.show()


In [ ]:
# Channel intensity histograms on the sanity batch only -- pre/post
# should be sharply right-skewed (sparse bright dots on near-zero
# background); structural should have a fatter tail (dendrites + somas).
sub = [get_patch(i) for i in SANITY_INDICES]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ch, name in enumerate(['pre (ch0)', 'post (ch1)', 'structural (ch2)']):
    vals = np.concatenate([p[ch].ravel() for p in sub])
    axes[ch].hist(vals, bins=80, log=True, color=['C2', 'C3', 'C0'][ch])
    axes[ch].set_title(f'{name}  (n={len(sub)} sanity patches)')
    axes[ch].set_xlabel('intensity'); axes[ch].set_ylabel('count (log)')
    axes[ch].grid(alpha=0.3)
fig.suptitle('Sanity batch -- channel intensity histograms', fontsize=11)
fig.tight_layout()
save_fig(fig, '02_sanity_channel_histograms')
plt.show()


## Calibrate the global Meijering threshold

Pool the Meijering response across `N_CALIBRATION_IMAGES` **full
reassembled images** (not patches) and run Otsu on the pooled
non-zero values. This avoids the patch-border artefact (a 27 px
rim per side at `sigma_max=9`) and gives a threshold representative
of how Meijering will be applied at full-image scale.

> **What you want to see**: a Otsu threshold somewhere in `[0.1, 0.5]`
> for real data. If near 0 the structural channel is suspect; if near
> 1 the Meijering response is degenerate (sigmas wrong for the data).


In [ ]:
rng = np.random.default_rng(SEED)
calib_image_indices = rng.choice(
    available_image_indices,
    size=min(N_CALIBRATION_IMAGES, len(available_image_indices)),
    replace=False,
).tolist()
logger.info(f'calibration images: {calib_image_indices}')

# Reassemble each calibration image (cached) and pull its full structural channel.
calib_full_struct = [
    get_full_image(int(i))[cfg.structural_channel] for i in calib_image_indices
]

global_thr = compute_global_meijering_threshold(
    calib_full_struct, sigmas=cfg.dendrite_sigmas, method='otsu',
)
logger.info(f'global Meijering Otsu threshold = {global_thr:.4f}')

# Pooled response distribution + chosen threshold (full-image responses)
pooled = []
for s in calib_full_struct:
    r = meijering_response(s, cfg.dendrite_sigmas)
    pooled.append(r[r > 0])
pooled = np.concatenate(pooled)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(pooled, bins=80, log=True, color='C0', alpha=0.7)
ax.axvline(global_thr, color='red', ls='--', label=f'Otsu = {global_thr:.4f}')
ax.set_xlabel('Meijering response (>0 only)')
ax.set_ylabel('count (log)')
ax.set_title(f'Pooled Meijering response across {len(calib_full_struct)} full images')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, '03_meijering_calibration')
plt.show()

cfg.dendrite_threshold = float(global_thr)


## Pipeline stages on the sanity batch

Per-stage diagnostic figures on the canonical sanity patches. The
first sanity index drives the per-step demos (one panel per step);
the full 3x3 pipeline figure is then plotted for *every* sanity
index so that all five canonical patches end up in the thesis.


In [ ]:
# Pick the canonical demo patch for the per-step figures.
demo_idx = SANITY_INDICES[0]
demo_name = patch_records[demo_idx]['filename']
demo = get_patch(demo_idx)
logger.info(f'demo patch: idx={demo_idx}  {demo_name}')


### Threshold sweeps (data-driven parameter picking)

Before running Step A, sweep candidate values for two thresholds on
the demo image's structural channel:

1. **`dendrite_threshold`** -- Otsu on the pooled Meijering response
   can collapse to a too-low value on pixelated data (background
   noise dominates). Compare Otsu to several percentiles and pick
   the one that highlights the visible dendrite tree without
   filling the whole field with speckle.
2. **`soma_intensity_percentile`** -- a too-aggressive percentile
   (`p=99`) can pick scattered punctate maxima inside dendrites,
   leaving zero connected components above `soma_min_area`. Lower
   percentiles let the broad soma fill survive as a single blob.

After picking values from these figures, **uncomment the override
cell below and clear the structural-mask cache** so downstream
panels recompute with the new parameters.


In [ ]:
# Dendrite threshold sweep on the demo image.
demo_image_index = int(patch_records[demo_idx]['image_index'])
demo_struct_channel = get_full_image(demo_image_index)[cfg.structural_channel]
demo_response = meijering_response(demo_struct_channel, cfg.dendrite_sigmas)
nz = demo_response[demo_response > 0]

thr_candidates = {
    'otsu (current)': float(cfg.dendrite_threshold),
    'p90': float(np.percentile(nz, 90)),
    'p95': float(np.percentile(nz, 95)),
    'p98': float(np.percentile(nz, 98)),
    'p99': float(np.percentile(nz, 99)),
}

print(f"{'method':>20s} | {'thr':>8s} | {'mask frac':>10s}")
for name, thr in thr_candidates.items():
    frac = float((demo_response > thr).mean())
    print(f'{name:>20s} | {thr:8.4f} | {frac:10.2%}')

fig, axes = plt.subplots(1, len(thr_candidates), figsize=(4 * len(thr_candidates), 4))
for ax, (name, thr) in zip(axes, thr_candidates.items()):
    mask = demo_response > thr
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'{name}\nthr={thr:.3f}  frac={mask.mean():.1%}', fontsize=9)
    ax.axis('off')
fig.suptitle(
    f'Dendrite threshold sweep  (image_index={demo_image_index}, '
    f'sigmas={list(cfg.dendrite_sigmas)})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '03b_dendrite_threshold_sweep')
plt.show()


In [ ]:
# Soma percentile sweep on the demo image.
# Pure single-knob exploration: raw intensity threshold + size filter
# (no closing / fill_holes). Shows the mask as a coloured overlay on the
# structural channel so you can see which somas survive each percentile.
from skimage.measure import label as cc_label, regionprops
from skimage.morphology import remove_small_objects

soma_pcts = [85.0, 90.0, 95.0, 99.0]
fig, axes = plt.subplots(1, len(soma_pcts), figsize=(4 * len(soma_pcts), 4))
print(
    f"{'pct':>6s} | {'thr':>8s} | {'#CC kept':>10s} | "
    f"{'max area':>10s} | {'mask frac':>10s}"
)
for ax, pct in zip(axes, soma_pcts):
    thr = float(np.percentile(demo_struct_channel, pct))
    bright = demo_struct_channel > thr
    keep = remove_small_objects(bright, min_size=cfg.soma_min_area)
    lbl = cc_label(keep)
    n_cc = int(lbl.max())
    max_area = int(max((p.area for p in regionprops(lbl)), default=0))
    print(
        f'{pct:6.1f} | {thr:8.4f} | {n_cc:10d} | '
        f'{max_area:10d} | {keep.mean():10.2%}'
    )
    show_mask_overlay(demo_struct_channel, keep, ax,
                      color=(1.0, 0.2, 0.2), alpha=0.5, vmax=0.5)
    ax.set_title(
        f'p={pct}, thr={thr:.3f}\n#CC={n_cc}  max_area={max_area}',
        fontsize=9,
    )
    ax.axis('off')
fig.suptitle(
    f'Soma percentile sweep  (image_index={demo_image_index}, '
    f'min_area={cfg.soma_min_area})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '04b_soma_percentile_sweep')
plt.show()


In [ ]:
# Override knobs from the sweeps above. Uncomment + edit, then run
# this cell -- the cache clear forces every downstream Step A / D /
# visual-review cell to recompute with the new parameters.
#
# cfg.dendrite_threshold        = thr_candidates['p98']
# cfg.soma_intensity_percentile = 90.0
# _struct_full_cache.clear()
# logger.info(
#     f'overrides applied: dendrite_threshold={cfg.dendrite_threshold:.4f}, '
#     f'soma_intensity_percentile={cfg.soma_intensity_percentile}; '
#     'struct cache cleared'
# )


### Step A -- structural mask (Meijering + soma) on the FULL image

Dendrites and somas are detected on the full reassembled image
containing the demo patch; the panels below show the full-image
computation with the demo-patch outline so you can see what the
patch is sliced from.


In [ ]:
from matplotlib.patches import Rectangle
from skimage.measure import label as cc_label

demo_rec = patch_records[demo_idx]
demo_image_index = int(demo_rec['image_index'])
demo_grid_row = int(demo_rec['grid_row'])
demo_grid_col = int(demo_rec['grid_col'])
patch_size = int(demo_rec['patch_size'])
y0 = demo_grid_row * patch_size
x0 = demo_grid_col * patch_size

full_struct = get_full_struct(demo_image_index)
full_struct_channel = full_struct['full_image'][cfg.structural_channel]

# Row 1: structural channel | dendrite mask (binary) | soma mask (binary)
# Row 2: Meijering response | dendrite mask OVERLAID on structural | soma mask OVERLAID on structural
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(full_struct_channel, cmap='gray', vmin=0, vmax=0.5)
axes[0, 0].set_title('structural channel (full)')

axes[0, 1].imshow(full_struct['dendrite_mask'], cmap='gray')
axes[0, 1].set_title(
    f'dendrite mask  (thr={cfg.dendrite_threshold:.3f}, '
    f'frac={full_struct["dendrite_mask"].mean():.2%})'
)

axes[0, 2].imshow(full_struct['soma_mask'], cmap='gray')
axes[0, 2].set_title(
    f'soma mask  (p={cfg.soma_intensity_percentile}, '
    f'min_area={cfg.soma_min_area}, '
    f'frac={full_struct["soma_mask"].mean():.2%})'
)

axes[1, 0].imshow(full_struct['meijering_response'], cmap='hot')
axes[1, 0].set_title(f'Meijering response\n(sigmas={list(cfg.dendrite_sigmas)})')

# Coloured overlays of each mask on the structural channel -- shows what
# is and what is NOT detected on the actual image (much more useful than
# the binary mask alone for spotting missed somas / dendrites).
n_soma_cc = int(cc_label(full_struct['soma_mask']).max())
show_mask_overlay(
    full_struct_channel, full_struct['dendrite_mask'],
    axes[1, 1], color=(0.2, 0.7, 1.0), alpha=0.45, vmax=0.5,
)
axes[1, 1].set_title('dendrite overlay on structural')

show_mask_overlay(
    full_struct_channel, full_struct['soma_mask'],
    axes[1, 2], color=(1.0, 0.2, 0.2), alpha=0.5, vmax=0.5,
)
axes[1, 2].set_title(f'soma overlay on structural  (#CC={n_soma_cc})')

for ax in axes.flat:
    ax.add_patch(Rectangle((x0, y0), patch_size, patch_size,
                           edgecolor='lime', facecolor='none', linewidth=1.5))
    ax.axis('off')
fig.suptitle(
    f'Step A: full-image structural mask  '
    f'(image_index={demo_image_index}, demo patch outlined in lime)',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '04_step_A_structural_mask')
plt.show()


In [ ]:
# Combined structural mask + dilation, sliced from the full-image computation.
struct_dict = get_patch_struct_slice(demo_idx)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(struct_dict['structural_mask'], cmap='gray')
axes[0].set_title(f'dendrite OR soma  (frac={struct_dict["structural_mask"].mean():.2%})')
axes[1].imshow(struct_dict['near_structural'], cmap='gray')
axes[1].set_title(
    f'after {cfg.structural_dilation}-px dilation  '
    f'(frac={struct_dict["near_structural"].mean():.2%})'
)
show_mask_overlay(demo.max(0), struct_dict['near_structural'],
                  axes[2], color=(0.2, 0.5, 1.0), alpha=0.4)
axes[2].set_title('overlay on composite')
fig.suptitle(
    f'Step A: structural mask + dilation, sliced from full image  '
    f'(idx={demo_idx})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '05_step_A_structural_dilation')
plt.show()


### Step B -- LoG blob detection on pre / post


In [ ]:
pre_blobs  = detect_blobs_log(demo[cfg.pre_channel],  cfg)
post_blobs = detect_blobs_log(demo[cfg.post_channel], cfg)
logger.info(f'pre  LoG candidates: {len(pre_blobs)}')
logger.info(f'post LoG candidates: {len(post_blobs)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_blob_overlay(demo[cfg.pre_channel],  pre_blobs,  axes[0], color='lime')
axes[0].set_title(f'pre   ch{cfg.pre_channel}  ({len(pre_blobs)} candidates)')
show_blob_overlay(demo[cfg.post_channel], post_blobs, axes[1], color='magenta')
axes[1].set_title(f'post  ch{cfg.post_channel}  ({len(post_blobs)} candidates)')
fig.suptitle(f'Step B: LoG blob detection  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '06_step_B_log_blobs')
plt.show()


### Step C -- per-blob z-score (SynQuant-lite)

For each LoG candidate compute the mean intensity inside the blob
disk and compare it against an *annular* background ring. The ratio
is converted to a z-score with a `sqrt(2 * ln n)` Cramer-style
correction; only blobs with `z >= cfg.zscore_threshold` survive.


In [ ]:
scored_pre  = score_blobs_zscore(demo[cfg.pre_channel],  pre_blobs,  cfg)
scored_post = score_blobs_zscore(demo[cfg.post_channel], post_blobs, cfg)

n_pre_kept  = sum(s['kept'] for s in scored_pre)
n_post_kept = sum(s['kept'] for s in scored_post)
logger.info(f'pre:  {n_pre_kept}/{len(scored_pre)} kept (z >= {cfg.zscore_threshold})')
logger.info(f'post: {n_post_kept}/{len(scored_post)} kept')

fig, ax = plt.subplots(figsize=(8, 4))
plot_zscore_histogram(scored_pre, scored_post, cfg.zscore_threshold, ax=ax)
fig.suptitle(f'Step C: z-score distribution  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '07_step_C_zscore_histogram')
plt.show()


In [ ]:
# Visual: kept (lime) vs rejected (red).
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_scored_blobs(demo[cfg.pre_channel],  scored_pre,  axes[0])
axes[0].set_title(f'pre  -- lime kept ({n_pre_kept}), red rejected')
show_scored_blobs(demo[cfg.post_channel], scored_post, axes[1])
axes[1].set_title(f'post -- lime kept ({n_post_kept}), red rejected')
fig.suptitle(f'Step C: z-score kept vs rejected  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '08_step_C_zscore_kept_vs_rejected')
plt.show()


### Threshold sweep

Sweep `zscore_threshold` over the canonical sanity batch and a few
additional random patches to see how recall changes. Use the curve
to pick a value that drops most red circles in section above without
removing the obvious lime ones.


In [ ]:
sweep_z = [3.0, 5.0, 7.0, 10.0, 15.0]
n_sweep_imgs = min(20, len(patch_records))
sweep_idx = sorted(set(SANITY_INDICES) | set(
    np.random.default_rng(SEED).choice(
        len(patch_records), n_sweep_imgs, replace=False,
    ).tolist()
))
# Pre-cache patches grouped by image so each image is reassembled at most once.
sweep_patches = {i: get_patch(i) for i in sorted(
    sweep_idx, key=lambda j: int(patch_records[j]['image_index'])
)}

rows = []
print(f'{"z":>5s} | {"pre kept":>10s} {"post kept":>10s}')
for z in sweep_z:
    cfg_t = BlobPseudoCfg(**{**cfg.__dict__, 'zscore_threshold': z})
    n_pre = n_post = 0
    for i in sweep_idx:
        p = sweep_patches[i]
        b_pre  = detect_blobs_log(p[cfg.pre_channel],  cfg_t)
        b_post = detect_blobs_log(p[cfg.post_channel], cfg_t)
        n_pre  += sum(s['kept'] for s in score_blobs_zscore(p[cfg.pre_channel],  b_pre,  cfg_t))
        n_post += sum(s['kept'] for s in score_blobs_zscore(p[cfg.post_channel], b_post, cfg_t))
    rows.append((z, n_pre, n_post))
    print(f'{z:5.1f} | {n_pre:10d} {n_post:10d}')

fig, ax = plt.subplots(figsize=(8, 4))
zs    = [r[0] for r in rows]
preks = [r[1] for r in rows]
posks = [r[2] for r in rows]
ax.plot(zs, preks, marker='o', color='C2', label='pre kept')
ax.plot(zs, posks, marker='s', color='C3', label='post kept')
ax.axvline(cfg.zscore_threshold, color='k', ls=':', alpha=0.6,
           label=f'cfg.zscore_threshold = {cfg.zscore_threshold}')
ax.set_xlabel('z-score threshold')
ax.set_ylabel('# blobs kept (sum across sweep patches)')
ax.set_title(f'z-score threshold sweep  (n={len(sweep_idx)} patches)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, '09_threshold_sweep')
plt.show()


### Step D -- full pipeline on every sanity patch

`generate_blob_pseudolabel` runs every step end-to-end. The 3x3 figure
shows every intermediate; the bottom-right is the final binary
pseudo-label. We render this figure for **every** sanity index so
all five canonical patches are saved for the thesis.


In [ ]:
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    p = get_patch(i)
    struct_slice = get_patch_struct_slice(i)
    label_mask, intermediates, stats = generate_blob_pseudolabel(
        p, cfg, precomputed_struct=struct_slice,
    )
    print(f'sanity[{j}]  idx={i}  {name}')
    for k, v in stats.items():
        print(f'  {k:24s}: {v}')
    fig, _ = show_pipeline_stages(p, intermediates, label_mask)
    fig.suptitle(
        f'Step D: full pipeline  sanity[{j}]  idx={i}  {name}\n'
        f'label_px={stats["px_label"]}  frac={stats["frac_label"]:.3%}',
        fontsize=11,
    )
    save_fig(fig, f'10_step_D_pipeline_sanity{j}_idx{i:05d}')
    plt.show()


## Full run -- save pseudo-labels for every patch

Each pseudo-label is saved as `(1, 128, 128)` float32 to
`OUTPUT_ROOT/<original_filename>.npy`. The `(1, ...)` leading axis
matches the `SegmentationPatchDataset` convention so this output
plugs directly into a downstream training notebook. Per-patch stats
are collected for the after-run analysis below.


In [ ]:
all_stats = []
with tqdm(total=len(patch_records), desc='generating pseudo-labels') as pbar:
    for img_idx in available_image_indices:
        # Reassemble + run Meijering/soma on the full image once.
        full_image, _ = reassemble_image(PATCH_ROOT, int(img_idx))
        full_struct = compute_fullimage_structural_mask(full_image, cfg)

        for pos in image_to_patch_positions[img_idx]:
            rec = patch_records[pos]
            name = rec['filename']
            ps = int(rec['patch_size'])
            y0 = int(rec['grid_row']) * ps
            x0 = int(rec['grid_col']) * ps
            patch = full_image[:, y0:y0 + ps, x0:x0 + ps]
            sl = (slice(y0, y0 + ps), slice(x0, x0 + ps))
            struct_slice = {
                'meijering_response': full_struct['meijering_response'][sl],
                'dendrite_mask':      full_struct['dendrite_mask'][sl],
                'soma_mask':          full_struct['soma_mask'][sl],
                'structural_mask':    full_struct['structural_mask'][sl],
                'near_structural':    full_struct['near_structural'][sl],
            }
            label, _, st = generate_blob_pseudolabel(
                patch, cfg, precomputed_struct=struct_slice,
            )
            out = label.astype(np.float32)[np.newaxis, ...]   # (1, 128, 128)
            np.save(OUTPUT_ROOT / name, out)
            st['filename'] = name
            st['image_index'] = int(img_idx)
            all_stats.append(st)
            pbar.update(1)

        # Free per-image arrays before reassembling the next one.
        del full_image, full_struct

# Persist per-patch stats next to the figures so the thesis figures
# can be re-derived without re-running the pipeline.
(save_dir / 'per_patch_stats.json').write_text(
    json.dumps([{k: (float(v) if isinstance(v, (np.floating,)) else v)
                  for k, v in s.items()} for s in all_stats], indent=2, default=str)
)
logger.info(f'saved {len(all_stats)} pseudo-label files to {OUTPUT_ROOT}')
logger.info(f'per_patch_stats.json written')


## After training plotting

Bigger visualisations after the full run, computed across many
patches. These are the figures intended for the thesis.


### Visual review on N random patches

Run the full pipeline on `N_VISUAL_REVIEW` random patches and plot
the final overlay. Use this to spot pathological patches.

Look for: red dots floating in empty regions (= structural mask too
permissive); dense red in a soma centre (= soma mask leaking inside);
no red at all (= z-score too strict or LoG sigmas wrong).


In [ ]:
rng = np.random.default_rng(SEED + 1)
review_idx = rng.choice(len(patch_records), size=min(N_VISUAL_REVIEW, len(patch_records)), replace=False)
# Sort by image_index to maximize cache reuse (one reassembly per image).
review_idx = sorted(review_idx, key=lambda j: int(patch_records[int(j)]['image_index']))

ncols = 4
nrows = int(np.ceil(len(review_idx) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.2 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, i in zip(axes, review_idx):
    name = patch_records[int(i)]['filename']
    p = get_patch(int(i))
    struct_slice = get_patch_struct_slice(int(i))
    lbl, _, st = generate_blob_pseudolabel(p, cfg, precomputed_struct=struct_slice)
    show_mask_overlay(p.max(0), lbl, ax, color=(1, 0.2, 0.2), alpha=0.6)
    short = name if len(name) < 36 else name[:16] + '...' + name[-16:]
    ax.set_title(
        f'idx={i}  {short}\n'
        f'label_px={st["px_label"]}  frac={st["frac_label"]:.3%}',
        fontsize=8,
    )
for ax in axes[len(review_idx):]:
    ax.axis('off')
fig.suptitle(
    f'Visual review  (n={len(review_idx)} random patches)', fontsize=12,
)
fig.tight_layout()
save_fig(fig, '11_visual_review_grid')
plt.show()


### Aggregate statistics

Global sanity checks on the dataset of pseudo-labels:

* **Label sparsity** -- typical synapse density is well under 1% of
  pixels. If most patches have >5% labelled, the structural mask is
  too permissive or the z-score threshold is too low.
* **Empty fraction** -- some patches will have no synapses (no
  dendrites in view). 10-30% empty is normal; 80%+ empty means LoG
  threshold is too high.
* **Per-patch blob counts** -- bimodal histograms suggest two
  populations (e.g. soma-rich vs. dendrite-only patches).


In [ ]:
label_pxs   = np.array([s['px_label']             for s in all_stats])
label_fracs = np.array([s['frac_label']           for s in all_stats])
n_pre_kept  = np.array([s['n_pre_kept']           for s in all_stats])
n_post_kept = np.array([s['n_post_kept']          for s in all_stats])
near_fracs  = np.array([s['frac_near_structural'] for s in all_stats])

summary = {
    'n_patches': len(all_stats),
    'patches_with_label': int((label_pxs > 0).sum()),
    'patches_with_label_frac': float((label_pxs > 0).mean()),
    'label_frac_median': float(np.median(label_fracs)),
    'label_frac_mean': float(label_fracs.mean()),
    'label_frac_max': float(label_fracs.max()),
    'pre_kept_median':  int(np.median(n_pre_kept)),
    'pre_kept_max':     int(n_pre_kept.max()),
    'post_kept_median': int(np.median(n_post_kept)),
    'post_kept_max':    int(n_post_kept.max()),
    'structural_area_median': float(np.median(near_fracs)),
    'structural_area_mean':   float(near_fracs.mean()),
}
(save_dir / 'aggregate_summary.json').write_text(json.dumps(summary, indent=2))
for k, v in summary.items():
    print(f'{k:30s}: {v}')
logger.info('aggregate_summary.json written')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].hist(label_fracs * 100, bins=40, edgecolor='k', alpha=0.7, color='C3')
axes[0, 0].set_xlabel('% pixels labelled')
axes[0, 0].set_title(
    f'Label sparsity per patch  '
    f'(median={np.median(label_fracs):.3%})'
)
axes[0, 1].hist(near_fracs * 100, bins=40, edgecolor='k', alpha=0.7, color='C0')
axes[0, 1].set_xlabel('% pixels in structural zone')
axes[0, 1].set_title(
    f'Structural-mask coverage  '
    f'(median={np.median(near_fracs):.2%})'
)
axes[1, 0].hist(n_pre_kept, bins=30, edgecolor='k', alpha=0.7, color='C2')
axes[1, 0].set_xlabel('# pre puncta')
axes[1, 0].set_title(f'Kept pre per patch  (median={int(np.median(n_pre_kept))})')
axes[1, 1].hist(n_post_kept, bins=30, edgecolor='k', alpha=0.7, color='C3')
axes[1, 1].set_xlabel('# post puncta')
axes[1, 1].set_title(f'Kept post per patch  (median={int(np.median(n_post_kept))})')
for ax in axes.flat:
    ax.grid(alpha=0.3)
fig.suptitle(
    f'Aggregate pseudo-label statistics  '
    fontsize=12,
)
fig.tight_layout()
save_fig(fig, '12_aggregate_statistics')
plt.show()


### Label sparsity vs structural coverage

Scatter of per-patch label fraction against structural-zone fraction.
A healthy run sits low on the y-axis (sparse labels) regardless of
structural coverage; a vertical band at high y indicates the
z-score is too permissive on dense-dendrite patches.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(near_fracs * 100, label_fracs * 100, s=8, alpha=0.4, color='C3')
ax.set_xlabel('% pixels in structural zone')
ax.set_ylabel('% pixels labelled')
ax.set_title(
    f'Label sparsity vs structural coverage  '
    f'(n={len(all_stats)})'
)
ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, '13_sparsity_vs_coverage')
plt.show()


## Notes & references

**Method choices (fact-checked, see *Patch-mode caveats* up top):**

* LoG `min_sigma=0.7, max_sigma=1.8` -- matches puncta diameter
  2-5 px at 107 nm/px (Lindeberg 1998, radius = sqrt(2)*sigma).
* `exclude_border=5` ~= 3*sigma_max -- drops detections in the
  unreliable ~5-px boundary rim of each 128x128 patch.
* Meijering `sigmas=range(2, 10)` -- matches dendrite radius 2-9 px
  (Meijering et al. 2004, sigma-as-radius).
* Global Meijering threshold via Otsu on pooled non-zero responses
  -- per-patch Otsu is unreliable when patches lack dendrites.
* Soma mask via 99th-percentile intensity threshold + 200-px area
  filter -- Meijering suppresses solid bright regions by design.
* Co-localisation dilation 2 px -- allows ~200 nm sub-pixel
  registration error.
* z-score 5.0 -- between SynQuant strict (10) and permissive (3).

**Papers, in order of relevance:**

* Wang, Y. et al. *SynQuant: an automatic tool to quantify synapses
  from fluorescence microscopy images.* Bioinformatics 36(5):1599
  (2020).
* Meijering, E. et al. *Design and validation of a tool for neurite
  tracing and analysis in fluorescence microscopy images.*
  Cytometry A 58:167 (2004).
* Lindeberg, T. *Feature detection with automatic scale selection.*
  IJCV 30:79 (1998).
* Frangi, A. F. et al. *Multiscale vessel enhancement filtering.*
  MICCAI 1998.
* Xiao, R. et al. *DDeep3M+: adaptive enhancement powered weakly
  supervised learning for neuron segmentation.* Neurophotonics 10(3)
  (2023).
* Fantuzzo, J. A. et al. *Intellicount: high-throughput quantification
  of fluorescent synaptic protein puncta by machine learning.*
  eNeuro (2017).
* Huang, Q. et al. *Weakly Supervised Learning of 3D Deep Network for
  Neuron Reconstruction.* Front. Neuroanat. 14:38 (2020).
